In [2]:
# ------------------------------------------------------------------ Paso 0: setup
import os, sys
from pathlib import Path
import pandas as pd
import numpy as np
import pyreadstat

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))
%load_ext autoreload
%autoreload 2
print(RAIZ)


c:\dev\proyecto1-ehpm-desempleo


In [3]:

# ------------------------------------------------------------------ Paso 1: fijar rutas

RAW       = Path("data/raw")
INTERIM   = Path("data/interim")
PROCESSED = Path("data/processed")
DB_DIR    = Path("db")
DOCS      = Path("docs")
for carpeta in (INTERIM, PROCESSED, DB_DIR, DOCS):
    carpeta.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 60)

In [22]:
# ------------------------------------------------------------------ Paso 2: ingesta
df, meta = pyreadstat.read_sav(RAW / "hogar_2025.sav")
print("Dimensiones:", df.shape)                         # (24534, 287)
print("Ejemplo de etiqueta:", meta.column_names_to_labels.get("CONDACT"))

Dimensiones: (28714, 342)
Ejemplo de etiqueta: Condicion de actividad


In [23]:
# ------------------------------------------------------------------ Paso 2: inventario
def construir_inventario(df, meta):
    return pd.DataFrame({
        "variable":  df.columns,
        "etiqueta":  [meta.column_names_to_labels.get(c, "") for c in df.columns],
        "dtype":     [str(df[c].dtype) for c in df.columns],
        "pct_na":    (df.isna().mean() * 100).round(1).to_numpy(),
        "n_unique":  [df[c].nunique(dropna=True) for c in df.columns],
    })

inv = construir_inventario(df, meta)
inv.to_csv(DOCS/ "inventario_2025.csv", index=False)

In [5]:
df.head()

,HOGAR,DOMINIO,DEPTO,COR_PRE,NUM_REC,NUM_HOG,V01,V02,V02OTRO,V03,V03OTRO,V04,V04OTRO,V05,V05OTRA,V06,V07,V07OTRO,V08,V08OTRA,V09,V10,V11,V12,H01_1,H01_2,H01_3,H01_4,H01_5,H01_6,...,OCUPAOP,RAMAOS,OCUPAOS,SALMIN,NSALMIN,NUMSALMIN,ACTIVIDA,TOTHRSOP,CATEGOS,ytraop,ytraos,ytrab,yotrf,ytothg,yperhg,quintil,yhpens,yhjubi,yhAlqu,yhOtros,yhayuf,yhayup,yhReme,yheduc,yhbono,ytrabhg,yotrfhg,ymophg,Pobreza,nbi
0,1.108081e+14,1.0,8.0,80941.0,10.0,1.0,1.0,1.0,,1.0,,3.0,,1.0,,1.0,1.0,,1.0,,6.0,3.0,10000.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,...,5.0,NaN,NaN,12539.68,1.0,2.0,3.0,72.0,99.0,12000.0,NaN,12000.0,NaN,24000.0,4800.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24000.0,NaN,24000.0,2.0,0.0
1,1.108081e+14,1.0,8.0,80941.0,10.0,1.0,1.0,1.0,,1.0,,3.0,,1.0,,1.0,1.0,,1.0,,6.0,3.0,10000.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2,1.108081e+14,1.0,8.0,80941.0,10.0,1.0,1.0,1.0,,1.0,,3.0,,1.0,,1.0,1.0,,1.0,,6.0,3.0,10000.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,...,5.0,NaN,NaN,12539.68,1.0,2.0,3.0,72.0,99.0,12000.0,NaN,12000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
3,1.108081e+14,1.0,8.0,80941.0,10.0,1.0,1.0,1.0,,1.0,,3.0,,1.0,,1.0,1.0,,1.0,,6.0,3.0,10000.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
4,1.108081e+14,1.0,8.0,80941.0,10.0,1.0,1.0,1.0,,1.0,,3.0,,1.0,,1.0,1.0,,1.0,,6.0,3.0,10000.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


In [24]:

# ------------------------------------------------------------------ Paso 3: selección
#Definimos un diccionario nombre_estándar → variable del archivo. Trabajar con 23 variables elegidas es mejor
# que con 287. La aserción falla temprano si algún nombre no existe. base es la tabla con la que seguimos.

COLS = {
    "departamento": "DEPTO",   # Departamento del encuestado  
    "edad":      "EDAD",      # años cumplidos (numérica)
    "sexo":      "SEXO",      # 1 Hombre, 2 Mujer
    "area":      "UR",        # 1 Urbana, 2 Rural
    "dominio":   "DOMINIO",   # 5 zonas de estimación
    "educacion": "ED05",      # nivel educativo más alto alcanzado
    "cond_act":  "CONDACT",   # 1 Ocupado, 2 Desocupado, 3 Fuera de la FT
    "tipo_inac": "TIPINAC",   # 2 = Desalentado
    "subocup":   "ACTIVIDA",  # 2 subocupado por tiempo, 3 por ingreso
    "categoria": "CATEGOP",   # categoría ocupacional
    "rama":      "RAMAOP",    # rama de la ocupación principal
    "aguinaldo": "OC620", # recibe aguinaldo
    "catorceavo": "OC621", # recibe catorceavo 
    "vacaciones": "OC622", # recibe bono de vacaciones (Esta y las dos anteriores para estimación de informalidad)
    "horas_principal": "TOTHRSOP",   # solo ocupación principal
    "ytrab":  "ytrab",               # ingreso por trabajo total
    "ytraop": "ytraop",      # Ingreso de la ocupación principal
    "ytraos": "ytraos",              # ingreso de la ocupación secundaria
    "desea_mas_horas": "CA522",      # ¿Hubiera querido trabajar más horas?
    "nsalmin":   "NSALMIN",   # tramo de salario mínimo
    "numsalmin": "NUMSALMIN", # tramo del salario mínimo con definiciones de subocupación
    "salmin":    "SALMIN",    # monto del salario mínimo aplicado
    "factor":    "FACTOR",    # factor de expansión
}
faltan = [v for v in COLS.values() if v not in df.columns]
assert not faltan, f"No están en el archivo: {faltan}"
base = df[list(COLS.values())].rename(columns={v: k for k, v in COLS.items()})
print(base.shape)
base.head(20)

(28714, 23)


,departamento,edad,sexo,area,dominio,educacion,cond_act,tipo_inac,subocup,categoria,rama,aguinaldo,catorceavo,vacaciones,horas_principal,ytrab,ytraop,ytraos,desea_mas_horas,nsalmin,numsalmin,salmin,factor
0,8.0,60.0,1.0,1.0,1.0,4.0,1.0,NaN,3.0,2.0,7.0,2.0,2.0,2.0,72.0,12000.0,12000.0,NaN,9.0,1.0,2.0,12539.68,491.247861
1,8.0,53.0,2.0,1.0,1.0,4.0,3.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,491.247861
2,8.0,30.0,1.0,1.0,1.0,4.0,1.0,NaN,3.0,2.0,7.0,2.0,2.0,2.0,72.0,12000.0,12000.0,NaN,9.0,1.0,2.0,12539.68,491.247861
3,8.0,25.0,2.0,1.0,1.0,NaN,3.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,491.247861
4,8.0,15.0,2.0,1.0,1.0,NaN,3.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,491.247861
5,8.0,44.0,1.0,1.0,1.0,5.0,1.0,NaN,1.0,4.0,7.0,NaN,NaN,NaN,48.0,30000.0,30000.0,NaN,2.0,3.0,5.0,12539.68,491.247861
6,8.0,43.0,2.0,1.0,1.0,5.0,1.0,NaN,1.0,1.0,15.0,1.0,1.0,2.0,40.0,16000.0,16000.0,NaN,2.0,2.0,4.0,15115.31,491.247861
7,8.0,16.0,2.0,1.0,1.0,NaN,3.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,491.247861
8,8.0,11.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,491.247861
9,8.0,10.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,491.247861


In [25]:
base = df[list(COLS.values())].rename(columns={v: k for k, v in COLS.items()})
crudo = base.copy()      # códigos originales, con los nombres estándar
print(base.shape)

(28714, 23)


In [26]:
# ------------------------------------------------------------------ Paso 4: tipado
#Pasamos a numéricas las que lo son, y a categorías con texto las de código. Así las tablas se leen solas y se evitan 
# promedios sin sentido sobre códigos. 

for c in ["edad", "horas_principal", "ytraop", "ytraos", "ytrab", "salmin", "factor"]:
    base[c] = pd.to_numeric(crudo[c], errors="coerce")

etiquetas = {
    "departamento":  {1: "Atlántida", 2: "Colón", 3: "Comayagua", 4: "Copán", 5: "Cortés", 6: "Choluteca",
          7: "El Paraíso", 8: "Francisco Morazán", 9: "Gracias a Dios", 10: "Intibucá",
          11: "Islas de la Bahía", 12: "La Paz", 13: "Lempira", 14: "Ocotepeque", 15: "Olancho",
          16: "Santa Bárbara", 17: "Valle", 18: "Yoro"},
    "sexo":     {1: "Hombre", 2: "Mujer"},
    "area":     {1: "Urbana", 2: "Rural"},
    "dominio":  {1: "Distrito Central", 2: "San Pedro Sula",
                 3: "Ciudades Medianas", 4: "Ciudades Pequeñas", 5: "Rural"},
    "educacion": {1: "Ninguno", 2: "Programa de alfabetización", 3: "Pre-básica (1-3)", 4: "Básica (1-9)", 5: "Media/Diversificado (1-4)",
                  6: "Técnico superior (1-3)", 7: "Superior no universitaria (1-4)", 8: "Superior universitaria (1-8)", 
                  9: "Especialidad (1-3)", 10: "Maestria (1-2)", 11: "Doctorado (1-4)", 99: "No sabe /  No responde"},
    "cond_act": {1: "Ocupado", 2: "Desocupado", 3: "Fuera de la fuerza"},
    "tipo_inac": {1: "Potencialmente Activos", 2: "Desalentados", 3: "Inactivos fuera de la fuerza de trabajo potencial"},
    "subocup": {1: "Sin problemas de empleo", 2: "Subocupados por tiempo de trabajo", 3: "Subocupados por ingreso"},
    "desea_mas_horas": {1: "Sí", 2: "No", 9: "No sabe"},
    "categoria" : {1: "Empleado u obrero público", 2: "Empleado u obrero privado", 3: "Empleado doméstico", 4: "Cuenta propia", 
                    5: "Aprendiz", 6: "Trabajador familiar", 7: "Contratista dependiente", 8: "No declaró categoría"},
    "rama" : {1: "Agricultura, ganadería, silvicultura y pesca", 2: "Explotacion de minas y canteras", 3: "Manufacturera", 
              4: "Suministro de electricidad, gas, vapor y aire acondicionado", 5: "Suministro de agua evacuación de aguas residuales, gestión de desechos y descontaminación", 
              6: "Construcción", 7: "Comercio al por mayor y al por menor reparación de vehículos automotores y motocicletas", 
              8: "Transporte y almacenamiento", 9: "Actividades de alojamiento y de servicio de comidas", 10: "Información y comunicaciones", 
              11: "Actividades financieras y de seguros", 12: "Actividades inmobiliarias", 13: "Actividades profesionales, científicas y técnicas", 
              14: "Actividades de servicios administrativos y de apoyo", 15: "Administración pública y defensa planes de seguridad social de afiliación obligatoria", 
              16: "Enseñanza", 17: "Actividades de atención de la salud humana y de asistencia social", 18: "Actividades artísticas, de entretenimiento y recreativas",
              19: "Otras actividades de servicios", 20: "Actividades de los hogares como empleadores actividades no diferenciadas de los hogares como productores de bienes y ser",
              21: "Actividades de organizaciones y órganos extraterritoriales", 22: "Rama sin especificar", 23: "Busca trabajo por primera vez",
              99: "Ns/Nr"},
    "aguinaldo": {1: "Sí", 2: "No"},
    "catorceavo": {1: "Sí", 2: "No"},
    "vacaciones": {1: "Sí", 2: "No"},
    "nsalmin": {1: "De menos de 1 salario", 2: "De 1 a 2 salarios", 3: "De 2 a 3 salarios",
            4: "De 3 a 4 salarios", 5: "De 4 y más salarios", 99: "No declaran ingresos"},

    "numsalmin": {1: "Menos de 1 salario y trabaja <36 horas", 2: "Menos de 1 salario y trabaja >=36 horas",
              3: "Menos de 1 salario y no declara horas", 4: "De 1 a 2 salarios", 5: "De 2 a 3 salarios",
              6: "De 3 a 4 salarios", 7: "De 4 salarios y más", 9: "No declara ingresos"},
}
for c, mapa in etiquetas.items():
    base[c] = crudo[c].map(mapa).astype("category")
base.dtypes

departamento       category
edad                float64
sexo               category
area               category
dominio            category
educacion          category
cond_act           category
tipo_inac          category
subocup            category
categoria          category
rama               category
aguinaldo          category
catorceavo         category
vacaciones         category
horas_principal     float64
ytrab               float64
ytraop              float64
ytraos              float64
desea_mas_horas    category
nsalmin            category
numsalmin          category
salmin              float64
factor              float64
dtype: object

## 4 Limpieza de faltantes, centinelas y verificación de rangos

In [10]:
#----------------------------- Revisar el perfil de las variables-------------------------------
def perfil(c):
    orig = COLS[c]
    s = base[c]
    print(c, '|', orig, '|', meta.column_names_to_labels.get(orig))
    print('etiquetas:', meta.variable_value_labels.get(orig))
    print(s.describe())
    print(s.value_counts(dropna=False).head(12))
    if pd.api.types.is_numeric_dtype(s):
        print('mayores:', s.nlargest(5).tolist())
    else:
        print('tipo:', s.dtype)

for c in base.columns:
    perfil(c)
    print('-' * 60)

departamento | DEPTO | Departamento
etiquetas: {1.0: 'Atlántida', 2.0: 'Colón', 3.0: 'Comayagua', 4.0: 'Copán', 5.0: 'Cortes', 6.0: 'Choluteca', 7.0: 'El Paraíso', 8.0: 'Francisco Morazán', 9.0: 'Gracias a Dios', 10.0: 'Intubucá', 11.0: 'Isla de la Bahía', 12.0: 'La Paz', 13.0: 'Lempira', 14.0: 'Ocotepeque', 15.0: 'Olancho', 16.0: 'Santa Bárbara', 17.0: 'Valle', 18.0: 'Yoro'}
count      28714
unique        16
top       Cortés
freq        7899
Name: departamento, dtype: object
departamento
Cortés               7899
Francisco Morazán    5293
Yoro                 1684
El Paraíso           1561
Olancho              1516
Santa Bárbara        1467
Comayagua            1395
Choluteca            1359
Lempira              1125
Copán                1109
Atlántida             998
Colón                 864
Name: count, dtype: int64
tipo: category
------------------------------------------------------------
edad | EDAD | Edad
etiquetas: None
count    28438.000000
mean        32.416590
std         2

In [46]:
pd.crosstab(df['TIPINAC'], base['cond_act'], dropna=False)

cond_act,Desocupado,Fuera de la fuerza,Ocupado
TIPINAC,,,
NaN,593,0,11616
1.0,0,74,0
2.0,0,362,0
3.0,0,8308,0


In [11]:
# 1) ¿el mapeo creó vacíos que antes no existían?
for c, orig in COLS.items():
    antes, despues = df[orig].isna().sum(), base[c].isna().sum()
    if antes != despues:
        print(c, 'vacíos antes:', antes, 'después:', despues)

# 2) ¿hay códigos en los datos sin etiqueta en el archivo?
for c, orig in COLS.items():
    etq = meta.variable_value_labels.get(orig)
    if etq:
        sin_etiqueta = set(df[orig].dropna().unique()) - set(etq)
        if sin_etiqueta:
            print(c, 'códigos sin etiqueta:', sin_etiqueta)

tipo_inac códigos sin etiqueta: {np.float64(3.0)}


### Función de limpieza 

In [13]:

# 1) Rangos plausibles: fuera de ellos, el valor se vuelve vacío (NaN)
NUMERICAS = {
    "edad":            (0, 110),
    "horas_principal": (0, 168),        # una semana tiene 168 horas
    "ytrab":           (0, np.inf),
    "ytraop":          (0, np.inf),
    "ytraos":          (0, np.inf),
    "salmin":          (0.01, np.inf),
    "factor":          (0.0001, np.inf),
}

# 2) Centinelas: textos que significan "no sé / no declaró", no un dato real
#    Deben ser idénticos a los de tu diccionario `etiquetas`
CENTINELAS = {
    "educacion":       ["No sabe /  No responde"],   # doble espacio, como en tu diccionario
    "desea_mas_horas": ["No sabe"],
    "nsalmin":         ["No declaran ingresos"],
    "numsalmin":       ["No declara ingresos"],
    "categoria":       ["No declaró categoría"],
    "rama":            ["Ns/Nr", "Rama sin especificar", "Busca trabajo por primera vez"],
}

def limpiar(base, crudo):
    """Devuelve (base_limpia, reporte). No modifica `base` ni `crudo`."""
    b = base.copy()
    ocupado = b["cond_act"].astype(str).eq("Ocupado")
    filas = []

    for c in b.columns:
        # a) datos que existían en el archivo y se perdieron al tipar
        perdidos = int((crudo[c].notna() & b[c].isna()).sum())
        n_cent = 0
        n_rango = 0

        # b) centinelas -> NaN
        if c in CENTINELAS:
            n_cent = int(b[c].isin(CENTINELAS[c]).sum())
            existentes = [x for x in CENTINELAS[c] if x in b[c].cat.categories]
            b[c] = b[c].cat.remove_categories(existentes)   # esos valores quedan en NaN

        # c) fuera de rango -> NaN
        if c in NUMERICAS:
            lo, hi = NUMERICAS[c]
            fuera = b[c].notna() & ~b[c].between(lo, hi)
            n_rango = int(fuera.sum())
            b.loc[fuera, c] = np.nan

        filas.append({
            "variable": c,
            "tipo": str(b[c].dtype),
            "perdidos_conversion": perdidos,
            "centinelas": n_cent,
            "fuera_de_regla": n_rango,
            "pct_na_total": round(b[c].isna().mean() * 100, 1),
            "pct_na_en_ocupados": round(b.loc[ocupado, c].isna().mean() * 100, 1),
        })

    return b, pd.DataFrame(filas)

In [14]:
#---------------------------- Generar base limpia y reporte-------------------------------#

base_limpia, reporte = limpiar(base, crudo)
reporte

,variable,tipo,perdidos_conversion,centinelas,fuera_de_regla,pct_na_total,pct_na_en_ocupados
0,departamento,category,0,0,0,0.0,0.0
1,edad,float64,0,0,0,1.0,0.0
2,sexo,category,0,0,0,0.0,0.0
3,area,category,0,0,0,0.0,0.0
4,dominio,category,0,0,0,0.0,0.0
5,educacion,category,0,234,0,28.9,6.2
6,cond_act,category,0,0,0,27.0,0.0
7,tipo_inac,category,0,0,0,69.5,100.0
8,subocup,category,0,0,0,59.5,0.0
9,categoria,category,0,0,0,58.5,0.0


In [ ]:
# ----------------------------------Comprobar hallazgos----------------------------------#

oc = base_limpia["cond_act"].eq("Ocupado")

print("1) Con categoría pero NO ocupados")
x = base_limpia[~oc & base_limpia["categoria"].notna()]
print(len(x)); print(x["cond_act"].value_counts(dropna=False)); print(x["edad"].describe())

print("\n2) Edad de quienes no tienen condición de actividad")
print(base_limpia.loc[base_limpia["cond_act"].isna(), "edad"].describe())

print("\n3) Prestaciones vacías según categoría (ocupados)")
print(pd.crosstab(base_limpia.loc[oc, "categoria"], base_limpia.loc[oc, "aguinaldo"].isna(), margins=True))

print("\n4) ¿Sin ingreso y sin respuesta de horas son las mismas personas?")
print(pd.crosstab(base_limpia.loc[oc, "ytrab"].isna(), base_limpia.loc[oc, "desea_mas_horas"].isna()))

print("\n5) Ocupados sin educación")
e = base_limpia[oc & base_limpia["educacion"].isna()]
print(len(e)); print(e["edad"].describe())

print("\n6) Departamentos presentes")
print(base_limpia["departamento"].value_counts(sort=False))

1) Con categoría pero NO ocupados
307
cond_act
NaN                   306
Fuera de la fuerza      1
Desocupado              0
Ocupado                 0
Name: count, dtype: int64
count    307.000000
mean      12.684039
std        1.791731
min        5.000000
25%       12.000000
50%       13.000000
75%       14.000000
max       26.000000
Name: edad, dtype: float64

2) Edad de quienes no tienen condición de actividad
count    7485.000000
mean        7.980895
std         3.953401
min         1.000000
25%         5.000000
50%         8.000000
75%        11.000000
max        14.000000
Name: edad, dtype: float64

3) Prestaciones vacías según categoría (ocupados)
aguinaldo                  False  True    All
categoria                                    
Aprendiz                      18     0     18
Contratista dependiente      124   376    500
Cuenta propia                  0  3464   3464
Empleado doméstico           308     1    309
Empleado u obrero privado   6117     2   6119
Empleado u obre

In [ ]:
#--------------------------- Comprobación final y banderas-------------------------#

b = base_limpia.copy()
b["pet"] = b["cond_act"].notna()
b["trabajo_infantil"] = b["cond_act"].isna() & b["categoria"].notna() & (b["edad"] < 15)

# Reglas que deben cumplirse
assert b.loc[b["pet"], "edad"].min() >= 15
assert b.loc[~b["pet"] & b["edad"].notna(), "edad"].max() <= 14
assert b["factor"].gt(0).all()
print("PET:", b["pet"].sum(), "| trabajo infantil:", b["trabajo_infantil"].sum())


PET: 20953 | trabajo infantil: 306


In [26]:
#---------------------------------- Guardar base limpia-----------------------------------#

ruta = RAIZ / "data" / "processed" / "ephpm_2025_limpia.parquet"
b.to_parquet(ruta, index=False)

# El reporte sí va a docs, porque es liviano y no contiene microdatos
reporte.to_csv(RAIZ / "docs" / "reporte_calidad_2025.csv", index=False, encoding="utf-8-sig")

print("Guardado:", ruta, "-", ruta.stat().st_size // 1024, "KB")

Guardado: c:\dev\proyecto1-ehpm-desempleo\data\processed\ephpm_2025_limpia.parquet - 212 KB


In [4]:
#----------------------------------- Cargar base limpia-----------------------------------#

base_limpia = pd.read_parquet(RAIZ / "data" / "processed" / "ephpm_2025_limpia.parquet")
base_limpia.head(10)

,departamento,edad,sexo,area,dominio,educacion,cond_act,tipo_inac,subocup,categoria,rama,aguinaldo,catorceavo,vacaciones,horas_principal,ytrab,ytraop,ytraos,desea_mas_horas,nsalmin,numsalmin,salmin,factor,pet,trabajo_infantil
0,Francisco Morazán,60.0,Hombre,Urbana,Distrito Central,Básica (1-9),Ocupado,NaN,Subocupados por ingreso,Empleado u obrero privado,Comercio al por mayor y al por menor reparació...,No,No,No,72.0,12000.0,12000.0,NaN,NaN,De menos de 1 salario,Menos de 1 salario y trabaja >=36 horas,12539.68,491.247861,True,False
1,Francisco Morazán,53.0,Mujer,Urbana,Distrito Central,Básica (1-9),Fuera de la fuerza,Inactivos fuera de la fuerza de trabajo potencial,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,491.247861,True,False
2,Francisco Morazán,30.0,Hombre,Urbana,Distrito Central,Básica (1-9),Ocupado,NaN,Subocupados por ingreso,Empleado u obrero privado,Comercio al por mayor y al por menor reparació...,No,No,No,72.0,12000.0,12000.0,NaN,NaN,De menos de 1 salario,Menos de 1 salario y trabaja >=36 horas,12539.68,491.247861,True,False
3,Francisco Morazán,25.0,Mujer,Urbana,Distrito Central,NaN,Fuera de la fuerza,Inactivos fuera de la fuerza de trabajo potencial,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,491.247861,True,False
4,Francisco Morazán,15.0,Mujer,Urbana,Distrito Central,NaN,Fuera de la fuerza,Inactivos fuera de la fuerza de trabajo potencial,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,491.247861,True,False
5,Francisco Morazán,44.0,Hombre,Urbana,Distrito Central,Media/Diversificado (1-4),Ocupado,NaN,Sin problemas de empleo,Cuenta propia,Comercio al por mayor y al por menor reparació...,NaN,NaN,NaN,48.0,30000.0,30000.0,NaN,No,De 2 a 3 salarios,De 2 a 3 salarios,12539.68,491.247861,True,False
6,Francisco Morazán,43.0,Mujer,Urbana,Distrito Central,Media/Diversificado (1-4),Ocupado,NaN,Sin problemas de empleo,Empleado u obrero público,Administración pública y defensa planes de seg...,Sí,Sí,No,40.0,16000.0,16000.0,NaN,No,De 1 a 2 salarios,De 1 a 2 salarios,15115.31,491.247861,True,False
7,Francisco Morazán,16.0,Mujer,Urbana,Distrito Central,NaN,Fuera de la fuerza,Inactivos fuera de la fuerza de trabajo potencial,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,491.247861,True,False
8,Francisco Morazán,11.0,Hombre,Urbana,Distrito Central,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,491.247861,False,False
9,Francisco Morazán,10.0,Hombre,Urbana,Distrito Central,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,491.247861,False,False


## Indicadores del mercado laboral


### Indicadores base: PET, PEA, ocupados, desocupados, inactivos, desalentados, y las tasas de desempleo, participación y ocupación.

In [5]:
# 1. Banderas (se crean aquí, a partir de cond_act; no se guardan hasta validarlas)

base_limpia["pet"]        = base_limpia["cond_act"].notna()
base_limpia["ocupado"]    = base_limpia["cond_act"].eq("Ocupado")
base_limpia["desocupado"] = base_limpia["cond_act"].eq("Desocupado")
base_limpia["pea"]        = base_limpia["ocupado"] | base_limpia["desocupado"]
base_limpia["inactivo"]   = base_limpia["cond_act"].eq("Fuera de la fuerza")
base_limpia["desalentado"] = base_limpia["tipo_inac"].eq("Desalentados")

base_limpia["grupo_edad"] = pd.cut(base_limpia["edad"], bins=[14, 24, 34, 44, 54, 64, 120],
                         labels=["15-24", "25-34", "35-44", "45-54", "55-64", "65 y más"])

In [6]:
# 2. Función de indicadores ponderados. Cada cifra suma factor; el conteo sin ponderar 
# (n_*) sirve para saber si una celda tiene muy pocos casos.


def indicadores(df, por=None):
    def calc(g):
        w = g["factor"]
        pet, pea = w[g.pet].sum(), w[g.pea].sum()
        oc, de   = w[g.ocupado].sum(), w[g.desocupado].sum()
        ina, dsa = w[g.inactivo].sum(), w[g.desalentado].sum()
        return pd.Series({
            "pet": pet, "pea": pea, "ocupados": oc, "desocupados": de,
            "inactivos": ina, "desalentados": dsa,
            "tda_%": de / pea * 100 if pea else np.nan,    # desempleo abierto
            "tgp_%": pea / pet * 100 if pet else np.nan,   # participación
            "t_ocup_%": oc / pet * 100 if pet else np.nan, # ocupación
            "n_pet": int(g.pet.sum()), "n_desocupados": int(g.desocupado.sum()),
        })
    if por is None:
        return calc(df).to_frame("Nacional").T
    filas = {k: calc(g) for k, g in df.groupby(por, observed=True)}
    return pd.DataFrame(filas).T.rename_axis(por)

indicadores(base_limpia)                    # nacional

,pet,pea,ocupados,desocupados,inactivos,desalentados,tda_%,tgp_%,t_ocup_%,n_pet,n_desocupados
Nacional,7.363421e+06,4.286430e+06,4.075415e+06,211014.64028,3.076991e+06,119999.327126,4.922853,58.212478,55.346763,20953.0,593.0


In [7]:
3. # Principales cortes de los indicadores

# 1) Educación agrupada para el tablero (propuesta; ajústala si quieres otros grupos)
GRUPO_EDUC = {
    "Ninguno": "Sin educación formal", "Programa de alfabetización": "Sin educación formal",
    "Pre-básica (1-3)": "Básica", "Básica (1-9)": "Básica",
    "Media/Diversificado (1-4)": "Media",
    "Técnico superior (1-3)": "Superior", "Superior no universitaria (1-4)": "Superior",
    "Superior universitaria (1-8)": "Superior", "Especialidad (1-3)": "Superior",
    "Maestria (1-2)": "Superior", "Doctorado (1-4)": "Superior",
}
orden_educ = pd.CategoricalDtype(["Sin educación formal", "Básica", "Media", "Superior"])
base_limpia["educ_grupo"] = base_limpia["educacion"].map(GRUPO_EDUC).astype(orden_educ)

# 2) Mostrar los vacíos como «Sin dato» en lugar de perderlos
def con_sin_dato(s):
    if isinstance(s.dtype, pd.CategoricalDtype) and "Sin dato" not in s.cat.categories:
        s = s.cat.add_categories("Sin dato")
    return s.fillna("Sin dato")

# 3) Una sola tabla larga con todos los cortes
CORTES = ["sexo", "grupo_edad", "departamento", "area", "dominio", "educ_grupo", "educacion"]

def tabla_cortes(df, cortes):
    partes = []
    for c in cortes:
        t = indicadores(df.assign(**{c: con_sin_dato(df[c])}), c)
        t = t.reset_index().rename(columns={c: "categoria"})
        t.insert(0, "corte", c)
        partes.append(t)
    return pd.concat(partes, ignore_index=True)

tabla = tabla_cortes(base_limpia, CORTES)
tabla.round(1)

,corte,categoria,pet,pea,ocupados,desocupados,inactivos,desalentados,tda_%,tgp_%,t_ocup_%,n_pet,n_desocupados
0,sexo,Hombre,3346465.9,2556378.1,2462889.4,93488.7,790087.8,37657.3,3.7,76.4,73.6,9626.0,258.0
1,sexo,Mujer,4016955.4,1730051.9,1612526.0,117525.9,2286903.5,82342.0,6.8,43.1,40.1,11327.0,335.0
2,grupo_edad,15-24,1710093.0,841194.7,770418.3,70776.4,868898.3,34062.2,8.4,49.2,45.1,4922.0,221.0
3,grupo_edad,25-34,1474474.8,1067369.1,995092.7,72276.4,407105.8,21929.6,6.8,72.4,67.5,4164.0,176.0
4,grupo_edad,35-44,1194414.0,847789.6,817617.0,30172.5,346624.4,23288.5,3.6,71.0,68.5,3507.0,86.0
5,grupo_edad,45-54,1111443.7,752484.5,729744.6,22739.8,358959.3,22370.6,3.0,67.7,65.7,3157.0,66.0
6,grupo_edad,55-64,814769.8,453971.4,442319.9,11651.5,360798.5,13562.5,2.6,55.7,54.3,2292.0,33.0
7,grupo_edad,65 y más,1058225.9,323620.8,320222.8,3398.0,734605.1,4786.0,1.0,30.6,30.3,2911.0,11.0
8,grupo_edad,Sin dato,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,0.0
9,departamento,Atlántida,374370.4,186978.4,175619.5,11358.8,187392.0,4184.3,6.1,49.9,46.9,705.0,14.0


In [ ]:
# -------------------------------- Guardar los valores del INE (2025) para fines de comparación -----------------------------

ine = pd.DataFrame({
    "PET":                [7363421, 3098079, 4265343],
    "PET hombre":         [3346466, 1469285, 1877181],
    "PET mujer":          [4016955, 1628793, 2388162],
    "Fuerza de trabajo":  [4286430, 1691224, 2595206],
    "FT hombre":          [2556378, 1152226, 1404152],
    "FT mujer":           [1730052,  538997, 1191055],
    "Tasa participación": [58.2, 54.6, 60.8],
    "TP masculina":       [76.4, 78.4, 74.8],
    "TP femenina":        [43.1, 33.1, 49.9],
    "Ocupados":           [4075415, 1633571, 2441845],
    "Asalariados":        [2441067,  908494, 1532573],
    "No asalariados":     [1634348,  725077,  909272],
    "Desocupados":        [211015, 57653, 153362],
    "TDA":                [4.9, 3.4, 5.9],
    "Subocup tiempo":     [8.0, 9.6, 7.0],
    "Subocup ingresos":   [33.4, 30.0, 35.6],
}, index=["Nacional", "Rural", "Urbana"]).T
ine.to_csv(RAIZ / "docs" / "referencia_ine_2025.csv", encoding="utf-8-sig")



In [10]:
#------------------------------------ Comparar cálculos propios contra los del INE---------------------------------#

def calc_ambito(df):
    w = df["factor"]
    H, M = df["sexo"].eq("Hombre"), df["sexo"].eq("Mujer")
    pet, pea, oc = w[df.pet].sum(), w[df.pea].sum(), w[df.ocupado].sum()
    des = w[df.desocupado].sum()
    sub_t = w[df.ocupado & df["subocup"].eq("Subocupados por tiempo de trabajo")].sum()
    sub_i = w[df.ocupado & df["subocup"].eq("Subocupados por ingreso")].sum()
    return {
        "PET": pet, "PET hombre": w[df.pet & H].sum(), "PET mujer": w[df.pet & M].sum(),
        "Fuerza de trabajo": pea, "FT hombre": w[df.pea & H].sum(), "FT mujer": w[df.pea & M].sum(),
        "Tasa participación": pea / pet * 100,
        "TP masculina": w[df.pea & H].sum() / w[df.pet & H].sum() * 100,
        "TP femenina":  w[df.pea & M].sum() / w[df.pet & M].sum() * 100,
        "Ocupados": oc, "Desocupados": des, "TDA": des / pea * 100,
        "Subocup tiempo": sub_t / oc * 100, "Subocup ingresos": sub_i / oc * 100,
    }

mio = pd.DataFrame({
    "Nacional": calc_ambito(base_limpia),
    "Rural":    calc_ambito(base_limpia[base_limpia["area"].eq("Rural")]),
    "Urbana":   calc_ambito(base_limpia[base_limpia["area"].eq("Urbana")]),
})
comp = pd.concat({"mío": mio, "INE": ine.loc[mio.index], "dif": mio - ine.loc[mio.index]}, axis=1)
comp.round(1)

mío                              INE             \
                     Nacional      Rural     Urbana   Nacional      Rural   
PET                 7363421.3  3098078.6  4265342.7  7363421.0  3098079.0   
PET hombre          3346465.9  1469285.3  1877180.6  3346466.0  1469285.0   
PET mujer           4016955.4  1628793.3  2388162.1  4016955.0  1628793.0   
Fuerza de trabajo   4286430.0  1691223.7  2595206.3  4286430.0  1691224.0   
FT hombre           2556378.1  1152226.4  1404151.7  2556378.0  1152226.0   
FT mujer            1730051.9   538997.3  1191054.6  1730052.0   538997.0   
Tasa participación       58.2       54.6       60.8       58.2       54.6   
TP masculina             76.4       78.4       74.8       76.4       78.4   
TP femenina              43.1       33.1       49.9       43.1       33.1   
Ocupados            4075415.4  1633570.6  2441844.8  4075415.0  1633571.0   
Desocupados          211014.6    57653.1   153361.6   211015.0    57653.0   
TDA                       4.9        3.4        5.9        4.9        3.4   
Subocup tiempo            8.0        9.6        7.0        8.0        9.6   
Subocup ingresos         33.4       30.0       35.6       33.4       30.0   

                                   dif               
                       Urbana Nacional Rural Urbana  
PET                 4265343.0      0.3  -0.4   -0.3  
PET hombre          1877181.0     -0.1   0.3   -0.4  
PET mujer           2388162.0      0.4   0.3    0.1  
Fuerza de trabajo   2595206.0     -0.0  -0.3    0.3  
FT hombre           1404152.0      0.1   0.4   -0.3  
FT mujer            1191055.0     -0.1   0.3   -0.4  
Tasa participación       60.8      0.0  -0.0    0.0  
TP masculina             74.8     -0.0   0.0    0.0  
TP femenina              49.9     -0.0  -0.0   -0.0  
Ocupados            2441845.0      0.4  -0.4   -0.2  
Desocupados          153362.0     -0.4   0.1   -0.4  
TDA                       5.9      0.0   0.0    0.0  
Subocup tiempo            7.0      0.0   0.0   -0.0  
Subocup ingresos         35.6     -0.0  -0.0    0.0

In [ ]:
# -------------------------------- Hallar qué categorías suman los asalariados del INE -------------------------------- #

from itertools import combinations
por_cat = base_limpia[base_limpia["ocupado"]].groupby("categoria", observed=True)["factor"].sum()
objetivo = 2441067
cats = por_cat.index.tolist()
for r in range(1, len(cats) + 1):
    for comb in combinations(cats, r):
        if abs(por_cat[list(comb)].sum() - objetivo) < 1:
            print("Asalariados =", comb)

Asalariados = ('Empleado doméstico', 'Empleado u obrero privado', 'Empleado u obrero público')


In [40]:
# ---------------------------------- Cortes con comprobaciones y bandera de celdas pequeñas ----------------------------#

w = base_limpia["factor"]
nac = {"pet": w[base_limpia.pet].sum(), "pea": w[base_limpia.pea].sum(), "ocupados": w[base_limpia.ocupado].sum(),
       "desocupados": w[base_limpia.desocupado].sum(), "inactivos": w[base_limpia.inactivo].sum(),
       "desalentados": w[base_limpia.desalentado].sum()}

tabla = tabla_cortes(base_limpia, CORTES)
tabla = tabla[tabla["n_pet"] > 0].copy()

for c in tabla["corte"].unique():
    t = tabla[tabla["corte"].eq(c)]
    for col, tot in nac.items():
        assert abs(t[col].sum() - tot) < 1, (c, col, t[col].sum(), tot)
    assert t["n_pet"].sum() == base_limpia["pet"].sum(), c

UMBRAL = 30
tabla["tda_confiable"] = tabla["n_desocupados"] >= UMBRAL
print(tabla.groupby("corte")["tda_confiable"].agg(confiables="sum", total="count"))

              confiables  total
corte                          
area                   2      2
departamento           3     16
dominio                5      5
educ_grupo             4      5
educacion              4     12
grupo_edad             5      6
sexo                   2      2


In [17]:
# --------------- Diagnosticar el «Sin dato» de educación (786.449 personas, 10,7% del PET) --------------- #

sd = base_limpia[base_limpia["pet"] & base_limpia["educacion"].isna()]
print(len(sd), round(sd["factor"].sum()))
for c in ["grupo_edad", "cond_act", "area", "sexo"]:
    print(sd.groupby(c, observed=True)["factor"].sum().round().to_frame("personas"), "\n")
#print(crudo.loc[sd.index, "educacion"].value_counts(dropna=False).head(10))

2154 786449
            personas
grupo_edad          
15-24       581032.0
25-34       114831.0
35-44        35569.0
45-54        29329.0
55-64        12252.0
65 y más     13436.0 

                    personas
cond_act                    
Desocupado           16289.0
Fuera de la fuerza  510361.0
Ocupado             259799.0 

        personas
area            
Rural   219524.0
Urbana  566925.0 

        personas
sexo            
Hombre  351966.0
Mujer   434483.0 

